# Week 44, Day 2: Temporal difference learning and dopamine

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w044/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: The TD model of a conditioning trial

Complete run_td: for each step t compute V(t) and V(t+1) from the features, the TD error delta[t] = r[t+1] + gamma V(t+1) - V(t), and update w by alpha delta[t] x[t]. Train for 300 trials.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
T, cs, rt = 40, 10, 30          # time steps, cue onset, reward time

def features():
    x = np.zeros((T + 1, T))
    for t in range(cs, T):
        x[t, t - cs] = 1         # complete serial compound: one feature per step after the cue
    return x

def run_td(trials=300, alpha=0.1, gamma=1.0, omit_last=False, early_last=False):
    w = np.zeros(T); x = features(); D = []
    for k in range(trials):
        r = np.zeros(T + 1); r[rt] = 1
        if k == trials - 1 and omit_last:
            r[rt] = 0
        if k == trials - 1 and early_last:
            r[rt] = 0; r[rt - 5] = 1
        delta = np.zeros(T)
        for t in range(T):
            ...
        D.append(delta)
    return w, np.array(D)

w, D = run_td()
print(f"first trial: error peaks at step {D[0].argmax()} (reward arrives at {rt});  last trial: peaks at step {D[-1].argmax()} (cue at {cs})")
plt.imshow(D, aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1); plt.xlabel("time step"); plt.ylabel("trial"); plt.colorbar(); plt.show()

In [ ]:
import numpy as np
def test_early():
    "on the first trial the error peaks at the reward"
    assert D[0].argmax() == rt - 1 and abs(D[0].max() - 1) < 1e-9
def test_late():
    "after training the error peaks at the cue and is near zero at the reward"
    assert D[-1].argmax() == cs - 1 and D[-1].max() > 0.9 and abs(D[-1][rt - 1]) < 0.05

test_early()
test_late()
print("All checks passed")

### Exercise: Omission and early reward

Run run_td with omit_last=True and with early_last=True, and plot the last trial's error for each next to the normal last trial.

In [ ]:
_, D_omit = ...
_, D_early = ...
for nm, d in [("normal", D[-1]), ("omitted", D_omit[-1]), ("early", D_early[-1])]:
    print(f"{nm:8s} cue {d[cs - 1]:+.2f}, early time {d[rt - 6]:+.2f}, usual reward time {d[rt - 1]:+.2f}")
fig, ax = plt.subplots(3, 1, sharex=True, figsize=(6, 5))
for a, (nm, d) in zip(ax, [("cue, reward", D[-1]), ("cue, no reward", D_omit[-1]), ("cue, early reward", D_early[-1])]):
    a.bar(range(T), d); a.set_ylabel(nm, rotation=0, ha="right")
ax[-1].set_xlabel("time step"); plt.show()

In [ ]:
import numpy as np
def test_omit():
    "omission: positive error at the cue and a dip of about -1 at the expected reward time"
    d = D_omit[-1]
    assert d[cs - 1] > 0.9 and d[rt - 1] < -0.9
def test_early_reward():
    "early reward: a burst at the early time and a dip at the usual time"
    d = D_early[-1]
    assert d[rt - 6] > 0.5 and d[rt - 1] < -0.5

test_omit()
test_early_reward()
print("All checks passed")

### Exercise: Reward probability

Write run_prob(p): like run_td but the reward arrives with probability p on each trial. Train for 1000 trials and average the cue response over the last 300 trials, and the reward-time response over the last 300 rewarded trials.

In [ ]:
rng = np.random.default_rng(2003)

def run_prob(p, trials=1000, alpha=0.05):
    w = np.zeros(T); x = features(); cue, rew = [], []
    for k in range(trials):
        rewarded = rng.random() < p
        r = np.zeros(T + 1); r[rt] = float(rewarded)
        delta = np.zeros(T)
        for t in range(T):
            delta[t] = r[t + 1] + x[t + 1] @ w - x[t] @ w
            w += alpha * delta[t] * x[t]
        if k >= trials - 300:
            ...
    return np.mean(cue), np.mean(rew)

cue_resp, rew_resp = {}, {}
for p in [0.25, 0.5, 0.75]:
    cue_resp[p], rew_resp[p] = run_prob(p)
    print(f"p = {p}: cue response {cue_resp[p]:.2f}, response to delivered reward {rew_resp[p]:.2f}")

In [ ]:
import numpy as np
def test_cue():
    "the average cue response grows with reward probability, close to p"
    assert all(abs(cue_resp[p] - p) < 0.1 for p in cue_resp)
def test_reward():
    "the average response on rewarded trials falls with p, close to 1 - p"
    assert all(abs(rew_resp[p] - (1 - p)) < 0.1 for p in rew_resp)

test_cue()
test_reward()
print("All checks passed")